In [2]:
!pip install transformers peft datasets accelerate -q

In [3]:
import json

dataset = [
    {"instruction": "list files", "output": "ls"},
    {"instruction": "list all files including hidden", "output": "ls -la"},
    {"instruction": "check disk usage", "output": "df -h"},
    {"instruction": "show running processes", "output": "ps aux"},
    {"instruction": "check memory usage", "output": "free -h"},
    {"instruction": "check network ports", "output": "ss -tuln"},
    {"instruction": "show current directory", "output": "pwd"},
    {"instruction": "check cpu info", "output": "cat /proc/cpuinfo"},
    {"instruction": "show system uptime", "output": "uptime"},
    {"instruction": "check logged in users", "output": "who"},
    {"instruction": "show environment variables", "output": "printenv"},
    {"instruction": "check os version", "output": "uname -a"},
    {"instruction": "show network interfaces", "output": "ip addr"},
    {"instruction": "check top cpu processes", "output": "ps aux --sort=-%cpu | head -10"},
    {"instruction": "list running services", "output": "systemctl list-units --type=service --state=running"},
    {"instruction": "show file permissions in tmp", "output": "ls -la /tmp"},
    {"instruction": "check active connections", "output": "ss -tupn"},
    {"instruction": "show cron jobs", "output": "crontab -l"},
    {"instruction": "check kernel version", "output": "uname -r"},
    {"instruction": "show disk partitions", "output": "lsblk"},
]

with open("dataset.json", "w") as f:
    json.dump(dataset, f, indent=2)

print(f"✅ Dataset saved — {len(dataset)} examples")

✅ Dataset saved — 20 examples


In [5]:
import torch
import json
import warnings
warnings.filterwarnings("ignore")

from transformers import (
    AutoTokenizer, AutoModelForCausalLM,
    TrainingArguments, Trainer,
    DataCollatorForLanguageModeling
)
from peft import LoraConfig, get_peft_model, TaskType
from datasets import Dataset

#Load dataset
with open("dataset.json") as f:
    raw = json.load(f)

def format_prompt(ex):
    return f"### Instruction: {ex['instruction']}\n### Command: {ex['output']}"

texts = [format_prompt(d) for d in raw]
dataset = Dataset.from_dict({"text": texts})
print(f" {len(dataset)} training samples ready")

# ── Load model
MODEL = "facebook/opt-125m"
print(f"\n Loading {MODEL}...")

tokenizer = AutoTokenizer.from_pretrained(MODEL)
tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    MODEL,
    torch_dtype=torch.float32,
    low_cpu_mem_usage=True
)
print(" Base model loaded")

# Inject LoRA adapters
lora_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    r=8,
    lora_alpha=32,
    lora_dropout=0.05,
    target_modules=["q_proj", "v_proj"],
    bias="none"
)
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

# Tokenize
def tokenize(batch):
    tokens = tokenizer(
        batch["text"],
        truncation=True,
        max_length=128,
        padding="max_length"
    )
    tokens["labels"] = tokens["input_ids"].copy()
    return tokens

tokenized = dataset.map(tokenize, batched=True, remove_columns=["text"])
print(" Tokenization done")

# Train
args = TrainingArguments(
    output_dir="./task_agent_model",
    num_train_epochs=5,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    fp16=False,
    logging_steps=5,
    save_steps=100,
    save_total_limit=1,
    report_to="none"
)

trainer = Trainer(
    model=model,
    args=args,
    train_dataset=tokenized,
    data_collator=DataCollatorForLanguageModeling(tokenizer, mlm=False)
)

print("\n Training started...")
trainer.train()
print(" Training complete!")

model.save_pretrained("./task_agent_model")
tokenizer.save_pretrained("./task_agent_model")
print(" Model saved → ./task_agent_model")

✅ 20 training samples ready

⏳ Loading facebook/opt-125m...


Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie model.decoder.embed_tokens.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


✅ Base model loaded
trainable params: 294,912 || all params: 164,143,104 || trainable%: 0.1797


Map:   0%|          | 0/20 [00:00<?, ? examples/s]

✅ Tokenization done

⏳ Training started...


Step,Training Loss
5,6.178725
10,5.225833
15,4.901074


✅ Training complete!
✅ Model saved → ./task_agent_model


In [6]:
import subprocess
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import PeftModel

# ── Load fine-tuned model ─────────────────────────────
print("⏳ Loading fine-tuned model...")
base = AutoModelForCausalLM.from_pretrained(
    "facebook/opt-125m",
    torch_dtype=torch.float32,
    low_cpu_mem_usage=True
)
tokenizer = AutoTokenizer.from_pretrained("./task_agent_model")
tokenizer.pad_token = tokenizer.eos_token
model = PeftModel.from_pretrained(base, "./task_agent_model")
model.eval()
print(" Model ready\n")

# ── Command lookup
COMMAND_MAP = {
    "list files": "ls",
    "list all files": "ls -la",
    "hidden files": "ls -la",
    "disk usage": "df -h",
    "disk space": "df -h",
    "running processes": "ps aux --sort=-%cpu | head -15",
    "processes": "ps aux --sort=-%cpu | head -15",
    "memory": "free -h",
    "ram": "free -h",
    "ports": "ss -tuln",
    "open ports": "ss -tuln",
    "network ports": "ss -tuln",
    "current directory": "pwd",
    "where am i": "pwd",
    "cpu info": "cat /proc/cpuinfo | head -20",
    "uptime": "uptime",
    "who": "who",
    "logged in": "who",
    "users": "who",
    "environment": "printenv",
    "env variables": "printenv",
    "os version": "uname -a",
    "system info": "uname -a && cat /etc/os-release 2>/dev/null",
    "kernel": "uname -r",
    "network": "ip addr",
    "interfaces": "ip addr",
    "connections": "ss -tupn",
    "active connections": "ss -tupn",
    "cron": "crontab -l 2>/dev/null || echo 'No cron jobs'",
    "cron jobs": "crontab -l 2>/dev/null || echo 'No cron jobs'",
    "partitions": "lsblk",
    "disk partitions": "lsblk",
    "file permissions": "ls -la /tmp",
    "tmp": "ls -la /tmp",
    "top processes": "ps aux --sort=-%cpu | head -10",
    "services": "systemctl list-units --type=service --state=running 2>/dev/null | head -15",
}

def lookup_command(instruction):
    """Fast lookup — check if any key phrase is in the instruction."""
    text = instruction.lower().strip()
    for key, cmd in COMMAND_MAP.items():
        if key in text:
            return cmd
    return None

# ── LLM fallback
def llm_predict_command(instruction):
    """Use fine-tuned LLM when lookup table has no match."""
    prompt = f"### Instruction: {instruction}\n### Command:"
    inputs = tokenizer(prompt, return_tensors="pt", max_length=64, truncation=True)
    with torch.no_grad():
        output = model.generate(
            **inputs,
            max_new_tokens=20,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id
        )
    decoded = tokenizer.decode(output[0], skip_special_tokens=True)
    if "### Command:" in decoded:
        return decoded.split("### Command:")[-1].strip().split("\n")[0].strip()
    return None

#  Safe command runner
def run_command(cmd):
    try:
        result = subprocess.run(
            cmd, shell=True, capture_output=True,
            text=True, timeout=10
        )
        out = result.stdout.strip() or result.stderr.strip()
        return out if out else "(no output)"
    except subprocess.TimeoutExpired:
        return "⚠️ Command timed out"
    except Exception as e:
        return f"⚠️ Error: {e}"

# Main Agent
def task_agent(instruction):
    print(f"\n{'='*55}")
    print(f"👤 USER: {instruction}")
    print('='*55)

    # Step 1: fast lookup
    command = lookup_command(instruction)
    source = "lookup table"

    # Step 2: LLM fallback
    if not command:
        command = llm_predict_command(instruction)
        source = "LLM prediction"

    if not command:
        print(" Could not determine a command for this instruction.")
        return

    print(f" PREDICTED COMMAND [{source}]: {command}")
    print(f"\n  Executing...")
    output = run_command(command)

    print(f"\n OUTPUT:\n{'-'*40}")
    print(output[:1000])
    print('-'*40)
    print('='*55)

# ── Demo
print(" LLM Task Automation Agent Ready!\n")

demo_queries = [
    "list files",
    "check disk usage",
    "show running processes",
    "check memory usage",
    "check network ports",
    "show system info",
    "check uptime",
    "show logged in users",
]

for q in demo_queries:
    task_agent(q)

⏳ Loading fine-tuned model...


Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie model.decoder.embed_tokens.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


✅ Model ready

🚀 LLM Task Automation Agent Ready!


👤 USER: list files
🤖 PREDICTED COMMAND [lookup table]: ls

⚙️  Executing...

📋 OUTPUT:
----------------------------------------
dataset.json
sample_data
task_agent_model
----------------------------------------

👤 USER: check disk usage
🤖 PREDICTED COMMAND [lookup table]: df -h

⚙️  Executing...

📋 OUTPUT:
----------------------------------------
Filesystem      Size  Used Avail Use% Mounted on
overlay         108G   22G   86G  21% /
tmpfs            64M     0   64M   0% /dev
shm             5.8G  4.0K  5.8G   1% /dev/shm
/dev/root       2.0G  1.2G  748M  63% /usr/sbin/docker-init
/dev/sda1        57G   23G   35G  41% /kaggle/input
tmpfs           6.4G  224K  6.4G   1% /var/colab
tmpfs           6.4G     0  6.4G   0% /proc/acpi
tmpfs           6.4G     0  6.4G   0% /proc/scsi
tmpfs           6.4G     0  6.4G   0% /sys/firmware
----------------------------------------

👤 USER: show running processes
🤖 PREDICTED COMMAND [lookup table]: 

In [7]:
print(" Task Automation Agent — Interactive Mode")
print("Type 'exit' to quit\n")
print("Try: list files / check disk usage / show processes")
print("     check memory / open ports / system info\n")

while True:
    user_input = input("You: ").strip()
    if user_input.lower() in ["exit", "quit", "q"]:
        print("👋 Goodbye!")
        break
    if user_input:
        task_agent(user_input)
        print("\nExecution successful. Go to the next cell.")

🤖 Task Automation Agent — Interactive Mode
Type 'exit' to quit

Try: list files / check disk usage / show processes
     check memory / open ports / system info

You: show processes

👤 USER: show processes
🤖 PREDICTED COMMAND [lookup table]: ps aux --sort=-%cpu | head -15

⚙️  Executing...

📋 OUTPUT:
----------------------------------------
USER         PID %CPU %MEM    VSZ   RSS TTY      STAT START   TIME COMMAND
root         964 63.2 24.7 4764348 3293576 ?     Sl   14:25   4:04 node /datalab/web/pyright/pyright-langserver.js --stdio --cancellationReceive=file:b52ec94b660346376291a2717032c2ea7a55cb8081
root         654 40.1 18.1 4149120 2405108 ?     Ssl  14:24   3:00 /usr/bin/python3 -m colab_kernel_launcher -f /root/.local/share/jupyter/runtime/kernel-feefd898-4f1e-41b8-8bf3-50d2e1f85723.json
root          64  3.4  0.0      0     0 ?        Z    14:22   0:20 [python3] <defunct>
root          86  1.3  1.1 409672 154008 ?       Sl   14:22   0:08 /usr/bin/python3 /usr/local/bin/jupyter